# Scratch — สร้าง Weekly Table เองจาก Daily โดยไม่ใช้ไฟล์ Weekly เดิม

**เป้าหมายของ notebook นี้:** ทำ daily → weekly ด้วยตัวเอง ตาม working process ที่ตกลงกันไว้ **โดยไม่เปิด/ไม่อ้างอิงไฟล์
`data/raw/weekly_df_final_for_modeling.csv` หรือ `data/processed/weekly_features.csv` เลยระหว่างสร้าง** เพื่อพิสูจน์ว่าเข้าใจ
กระบวนการรวมข้อมูลทั้งหมดจริง ไม่ใช่แค่เชื่อไฟล์ที่มีอยู่แล้ว

**หมายเหตุสำคัญ**: นี่คือ notebook แบบฝึกหัด (scratch) แยกต่างหาก **ไม่ใช่ส่วนหนึ่งของ pipeline หลัก** (Phase 0-15)
ไม่กระทบ ไม่แก้ไข ไม่แทนที่ไฟล์ weekly เดิมหรือ notebook 02-11 ใดๆ ทั้งสิ้น ผลลัพธ์จะถูกเซฟแยกไว้ที่
`data/interim/weekly_mine_standalone.csv` เท่านั้น

**Input**: `data/raw/FMCG_2022_2024.csv` (ข้อมูลดิบรายวัน) — ไฟล์เดียวที่ใช้ตลอด notebook นี้


## 1. โหลดข้อมูลดิบและทำความเข้าใจโครงสร้าง

โหลด `FMCG_2022_2024.csv` เข้ามาดูก่อนว่ามีกี่แถว คอลัมน์อะไรบ้าง ครอบคลุมช่วงเวลาไหน — ยังไม่แตะไฟล์ weekly ใดๆ เลยในขั้นนี้


In [1]:
import pandas as pd

daily = pd.read_csv('../data/raw/FMCG_2022_2024.csv', parse_dates=['date'])
print('shape:', daily.shape)
print()
print(daily.dtypes)
print()
print('ช่วงวันที่:', daily['date'].min(), '->', daily['date'].max())

shape: (190757, 14)

date               datetime64[ns]
sku                        object
brand                      object
segment                    object
category                   object
channel                    object
region                     object
pack_type                  object
price_unit                float64
promotion_flag              int64
delivery_days                int64
stock_available              int64
delivered_qty                int64
units_sold                   int64
dtype: object

ช่วงวันที่: 2022-01-21 00:00:00 -> 2024-12-31 00:00:00

## 2. กำหนด Group Key ที่ห้ามปนกัน

แต่ละแถวของข้อมูลดิบคือ "สินค้า 1 ตัว × ช่องทาง 1 ช่องทาง × ภูมิภาค 1 ภูมิภาค × 1 วัน" — เวลารวมเป็นรายสัปดาห์
**ต้อง groupby ตาม `sku`, `channel`, `region` ก่อนเสมอ** ห้ามรวมทั้งไฟล์เป็นก้อนเดียว ไม่งั้นยอดขายของสินค้าคนละตัว
คนละช่องทางจะถูกบวกปนกัน


In [1]:
n_groups = daily.groupby(['sku', 'channel', 'region']).ngroups
print('จำนวนกลุ่ม (sku x channel x region):', n_groups)
print('จำนวน sku:', daily.sku.nunique(), ' channel:', daily.channel.nunique(), ' region:', daily.region.nunique())

จำนวนกลุ่ม (sku x channel x region): 270
จำนวน sku: 30  channel: 3  region: 3

## 3. ตัดสินใจ Convention การตัดสัปดาห์ (บันทึกไว้ก่อนเขียนโค้ด)

เพราะไม่มีไฟล์เฉลยให้ยึด ต้องตัดสินใจเองและบันทึกไว้ชัดเจนว่าเลือกกฎอะไร:

- **สัปดาห์เริ่มวันจันทร์** (`W-MON`)
- **label ด้วยวันแรกของสัปดาห์** ไม่ใช่วันสุดท้าย (`label='left'`)
- **ปิดขอบซ้าย** หมายถึงวันจันทร์ของสัปดาห์นั้นนับรวมอยู่ในสัปดาห์นั้น ไม่ใช่สัปดาห์ก่อนหน้า (`closed='left'`)

นี่คือทางเลือกหนึ่งที่สมเหตุสมผล (ธุรกิจจำนวนมากเริ่มสัปดาห์งานวันจันทร์) — ไม่ได้อ้างอิงว่า "ต้อง" ตรงกับไฟล์เดิม
เพราะตั้งใจไม่เปิดไฟล์นั้นดูในขั้นตอนนี้


## 4. ตัดสินใจวิธีรวมแต่ละคอลัมน์

| คอลัมน์ | วิธีรวม | เหตุผล |
|---|---|---|
| `units_sold`, `delivered_qty` | sum | ปริมาณสะสมทั้งสัปดาห์ |
| `price_unit`, `delivery_days` | mean | ค่า ณ ช่วงเวลา ไม่ใช่ปริมาณสะสม |
| `promotion_flag` | mean | ได้ "สัดส่วนวันที่มีโปร" แทนที่จะเป็นแค่ true/false |
| `stock_available` | mean | ระดับสต็อกเฉลี่ยของสัปดาห์ |


## 5. สร้าง Weekly Table

ใช้ groupby ตาม group key จากข้อ 2 แล้ว resample ตามเวลาภายในแต่ละกลุ่ม ตาม convention จากข้อ 3-4


In [1]:
weekly_mine = (
    daily.groupby(['sku', 'channel', 'region'])
    .resample('W-MON', label='left', closed='left', on='date')
    .agg({
        'units_sold': 'sum',
        'delivered_qty': 'sum',
        'price_unit': 'mean',
        'stock_available': 'mean',
        'promotion_flag': 'mean',
        'delivery_days': 'mean',
    })
    .reset_index()
)
print('shape:', weekly_mine.shape)
weekly_mine.head(5)

shape: (32377, 10)

      sku   channel      region       date  units_sold  delivered_qty  price_unit  stock_available  promotion_flag  delivery_days
0  JU-021  Discount  PL-Central 2022-07-11         159           1048    5.060000       162.666667        0.166667       3.500000
1  JU-021  Discount  PL-Central 2022-07-18         149            871    3.522000       163.600000        0.400000       2.800000
2  JU-021  Discount  PL-Central 2022-07-25          99            944    4.556667       143.166667        0.000000       2.833333
3  JU-021  Discount  PL-Central 2022-08-01         122           1052    4.940000       157.333333        0.333333       4.500000
4  JU-021  Discount  PL-Central 2022-08-08         130            961    4.382000       158.800000        0.600000       3.000000

## 6. ตรวจสอบความถูกต้อง — โดยไม่มีไฟล์เฉลยให้เทียบ

เพราะตั้งใจไม่ใช้ไฟล์ weekly เดิม จึงต้องพิสูจน์ความถูกต้องด้วยวิธีอื่นแทนการเทียบไฟล์ตรงๆ — ทำ 5 การตรวจสอบ


### 6.1 ผลรวมทั้งก้อนต้องเท่ากันเป๊ะ

In [1]:
d_sum = daily['units_sold'].sum()
w_sum = weekly_mine['units_sold'].sum()
print('ผลรวม daily:', d_sum)
print('ผลรวม weekly:', w_sum)
print('ตรงกันไหม:', d_sum == w_sum)

ผลรวม daily: 3799824
ผลรวม weekly: 3799824
ตรงกันไหม: True

ผลรวมยอดขายทั้งหมดตรงกันเป๊ะ — ยืนยันว่าไม่มีข้อมูลหายหรือถูกนับซ้ำระหว่างการรวม

### 6.2 Convention ต้องสม่ำเสมอ (ทุกวันที่ในคอลัมน์ week ต้องเป็นวันจันทร์)

In [1]:
all_monday = (weekly_mine['date'].dt.dayofweek == 0).all()
print('ทุกแถวเริ่มวันจันทร์จริงไหม:', all_monday)

ทุกแถวเริ่มวันจันทร์จริงไหม: True

### 6.3 ไม่มี key ซ้ำ

In [1]:
dup = weekly_mine.duplicated(subset=['sku', 'channel', 'region', 'date']).sum()
print('จำนวนแถวที่ key ซ้ำ:', dup)

จำนวนแถวที่ key ซ้ำ: 0

### 6.4 จำนวนแถวต่อกลุ่มต้องสมเหตุสมผล

In [1]:
rows_per_group = weekly_mine.groupby(['sku', 'channel', 'region']).size()
rows_per_group.describe()

count    270.000000
mean     119.914815
std       22.287830
min       85.000000
25%       99.000000
50%      118.500000
75%      143.000000
max      155.000000
dtype: float64

จำนวนสัปดาห์ต่อกลุ่มอยู่ในช่วง 85-155 สัปดาห์ สอดคล้องกับข้อมูล 3 ปี (2022-2024 ≈ 156 สัปดาห์) และสอดคล้องกับที่รู้จาก
Phase 2 ว่า SKU แต่ละตัวเปิดตัวคนละช่วงเวลา (บาง SKU มีประวัติสั้นกว่าเพราะเพิ่งเปิดตัว) — ไม่มีตัวเลขที่ดูผิดปกติ


### 6.5 สุ่มตรวจด้วยมือ 1 ตัวอย่าง (ไม่พึ่งพาไฟล์ใดๆ เลย)

In [1]:
sample = daily[(daily.sku == 'MI-006') & (daily.channel == 'Retail') & (daily.region == 'PL-Central')].sort_values('date')
first_week_start = weekly_mine[
    (weekly_mine.sku == 'MI-006') & (weekly_mine.channel == 'Retail') & (weekly_mine.region == 'PL-Central')
].sort_values('date').iloc[0]['date']

manual_days = sample[(sample['date'] >= first_week_start) & (sample['date'] < first_week_start + pd.Timedelta(days=7))]
manual_sum = manual_days['units_sold'].sum()

code_sum = weekly_mine[
    (weekly_mine.sku == 'MI-006') & (weekly_mine.channel == 'Retail') & (weekly_mine.region == 'PL-Central')
    & (weekly_mine.date == first_week_start)
]['units_sold'].values[0]

print('สัปดาห์ที่เช็ค:', first_week_start.date())
print(manual_days[['date', 'units_sold']].to_string(index=False))
print()
print('รวมด้วยมือ:', manual_sum, ' | โค้ดคำนวณได้:', code_sum, ' | ตรงกันไหม:', manual_sum == code_sum)

สัปดาห์ที่เช็ค: 2022-01-17
      date  units_sold
2022-01-21           9
2022-01-22          14
2022-01-23          11

รวมด้วยมือ: 34  | โค้ดคำนวณได้: 34  | ตรงกันไหม: True

**ข้อสังเกต**: สัปดาห์แรกของกลุ่มนี้มีข้อมูลแค่ 3 วัน (21-23 ม.ค.) ไม่ครบ 7 วัน เพราะข้อมูลดิบเริ่มต้นวันที่ 21 ม.ค. 2022
ซึ่งไม่ใช่วันจันทร์ — ผลรวมยังถูกต้องตามที่มีข้อมูลจริง แต่เป็นสัปดาห์ที่ "ไม่ครบสัปดาห์" ตามธรรมชาติ ไม่ใช่ข้อผิดพลาด (ดูข้อ 7)


## 7. บันทึกผลลัพธ์

เซฟไฟล์แยกไว้ที่ `data/interim/` — **ไม่แทนที่หรือแก้ไขไฟล์ weekly เดิมใดๆ**


In [1]:
weekly_mine.to_csv('../data/interim/weekly_mine_standalone.csv', index=False)
print('บันทึกแล้ว: data/interim/weekly_mine_standalone.csv')
print('จำนวนแถว:', len(weekly_mine))

บันทึกแล้ว: data/interim/weekly_mine_standalone.csv
จำนวนแถว: 32377

## 8. ส่วนเสริม — เปรียบเทียบจำนวนแถวกับไฟล์เดิม (เพื่อความเข้าใจ ไม่ใช่เพื่อ validate)

ขั้นตอนนี้**เปิดไฟล์เดิมมาดูแค่จำนวนแถว**เพื่อความเข้าใจเชิงเปรียบเทียบเท่านั้น (ทำหลังจากสร้างและตรวจสอบไฟล์ของตัวเองเสร็จ
สมบูรณ์แล้วในข้อ 1-7 ข้างต้น — ไม่ได้ใช้ไฟล์เดิมช่วยสร้างหรือแก้ไขผลลัพธ์ของตัวเองแต่อย่างใด)


In [1]:
weekly_existing = pd.read_csv('../data/raw/weekly_df_final_for_modeling.csv', parse_dates=['week'])
print('จำนวนแถวของตัวเอง (mine):', len(weekly_mine))
print('จำนวนแถวไฟล์เดิม (existing):', len(weekly_existing))
print('ผลต่าง:', len(weekly_mine) - len(weekly_existing))

จำนวนแถวของตัวเอง (mine): 32377
จำนวนแถวไฟล์เดิม (existing): 31027
ผลต่าง: 1350

In [1]:
g_mine = weekly_mine[(weekly_mine.sku == 'MI-006') & (weekly_mine.channel == 'Retail') & (weekly_mine.region == 'PL-Central')]
g_exist = weekly_existing[(weekly_existing.sku == 'MI-006') & (weekly_existing.channel == 'Retail') & (weekly_existing.region == 'PL-Central')]

extra_weeks = sorted(set(g_mine.date) - set(g_exist.week))
print('สัปดาห์ที่มีในไฟล์ของตัวเอง แต่ไม่มีในไฟล์เดิม (ตัวอย่าง MI-006/Retail/PL-Central):')
print(extra_weeks)
print('จำนวน:', len(extra_weeks), ' x 270 กลุ่ม =', len(extra_weeks) * 270, '(ตรงกับผลต่างรวม 1,350 พอดี)')

สัปดาห์ที่มีในไฟล์ของตัวเอง แต่ไม่มีในไฟล์เดิม (ตัวอย่าง MI-006/Retail/PL-Central):
[Timestamp('2022-01-17 00:00:00'), Timestamp('2022-01-24 00:00:00'), Timestamp('2022-01-31 00:00:00'), Timestamp('2022-02-07 00:00:00'), Timestamp('2024-12-30 00:00:00')]
จำนวน: 5  x 270 กลุ่ม = 1350 (ตรงกับผลต่างรวม 1,350 พอดี)

**คำอธิบายผลต่าง (1,350 แถว = 5 สัปดาห์ × 270 กลุ่ม พอดี):** ไม่ใช่ข้อผิดพลาดในการรวมข้อมูล (ข้อ 6 พิสูจน์แล้วว่าผลรวม
ตรงกันเป๊ะ) แต่เกิดจาก**เหตุผลเชิงจุดประสงค์ของไฟล์**:

- ไฟล์เดิม (`weekly_df_final_for_modeling.csv`) มีคอลัมน์ `lag_1`, `lag_2`, `rolling_mean_4`, `rolling_std_4`, `momentum`,
  และ `target_next_week` อยู่แล้ว — การคำนวณ feature เหล่านี้ต้องการ**ประวัติสัปดาห์ก่อนหน้าอย่างน้อย 4 สัปดาห์** และ
  **ต้องมีสัปดาห์ถัดไปให้เป็น target** จึงเป็นไปได้ว่าไฟล์เดิม**ตัดสัปดาห์ต้น/ท้ายของแต่ละกลุ่มที่คำนวณ feature เหล่านี้ไม่ครบ
  ออกไปแล้ว** ก่อนจะเผยแพร่เป็นตารางสำหรับสร้างโมเดล
- ไฟล์ของเรา (ข้อ 1-7) เป็นแค่ตัวเลขรวมยอดขาย/ราคา/สต็อกดิบๆ ยังไม่มี lag/rolling/target จึง**ไม่มีเหตุผลต้องตัดสัปดาห์ต้น-ท้ายทิ้ง**
  เก็บไว้ครบทุกสัปดาห์ที่มีข้อมูลจริง รวมถึงสัปดาห์แรกที่ไม่ครบ 7 วัน (ตามที่เห็นในข้อ 6.5) และสัปดาห์สุดท้าย 30 ธ.ค. 2024
  (มีข้อมูลแค่ 1-2 วันก่อนข้อมูลดิบจะหมดที่ 31 ธ.ค. 2024)

**ข้อสรุป**: นี่คือความแตกต่างที่อธิบายได้และสมเหตุสมผล ไม่ใช่ bug — เป็นตัวอย่างที่ดีว่าทำไม "จำนวนแถวไม่ตรงกัน" ไม่ได้แปลว่า
"ผิด" เสมอไป ต้องเข้าใจที่มาของแต่ละไฟล์ก่อนสรุป


## 9. สรุป

| หัวข้อ | ผลลัพธ์ |
|---|---|
| ผลรวมยอดขาย (sum check) | ตรงกัน 100% (3,799,824 หน่วยทั้งคู่) |
| Convention สม่ำเสมอ | ทุกแถวเริ่มวันจันทร์จริง |
| Duplicate key | ไม่มี |
| จำนวนแถวต่อกลุ่ม | อยู่ในช่วงสมเหตุสมผล (85-155 สัปดาห์) |
| สุ่มตรวจด้วยมือ | ตรงกัน (ตัวอย่าง MI-006/Retail/PL-Central สัปดาห์แรก) |
| Output | `data/interim/weekly_mine_standalone.csv` — 32,377 แถว |
| ผลต่างจำนวนแถวกับไฟล์เดิม | 1,350 แถว อธิบายได้ชัดเจน (ไฟล์เดิมตัดสัปดาห์ต้น/ท้ายที่คำนวณ lag/rolling/target ไม่ครบออก) |

**ไฟล์ที่แตะ**: `data/raw/FMCG_2022_2024.csv` (อ่านอย่างเดียว), `data/raw/weekly_df_final_for_modeling.csv`
(อ่านอย่างเดียว เฉพาะข้อ 8 เพื่อเปรียบเทียบ), `data/interim/weekly_mine_standalone.csv` (สร้างใหม่)

**ไฟล์ที่ไม่แตะเลย**: `data/processed/weekly_features.csv`, `src/`, notebook 02-11 ทั้งหมด, และ pipeline หลักของโปรเจกต์
